In [ ]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("mdwaquarazam/microorganism-image-classification")

# print("Path to dataset files:", path)

In [2]:
import os
import cv2
import numpy as np
from glob import glob
from ripser import ripser
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA

# --- KONFIGURACJA ---
DATA_DIR = "./Micro_Organism"  # Aktualny folder, w którym są subfoldery (Amoeba, Euglena, itd.)
CLASSES = [
    "Amoeba", "Euglena", "Hydra", "Paramecium", 
    "Rod_bacteria", "Spherical_bacteria", "Spiral_bacteria", "Yeast"
]
MAX_POINTS = 300  # Subsampling dla Ripsera, aby działał błyskawicznie
FILTRATION_STEPS = 50
MAX_FILTRATION = 20.0  # Zależne od rozmiaru obrazu, 20.0 dla 64x64 jest zazwyczaj ok

def extract_point_cloud(image_path, max_points=MAX_POINTS):
    """Przekształca obraz w chmurę punktów 2D za pomocą detekcji krawędzi."""
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    
    # Standaryzacja rozmiaru obrazu
    img = cv2.resize(img, (64, 64))
    
    # Ekstrakcja krawędzi
    edges = cv2.Canny(img, threshold1=100, threshold2=200)
    
    # Pobranie współrzędnych punktów krawędzi (x, y)
    y, x = np.where(edges > 0)
    points = np.column_stack((x, y))
    
    # Subsampling, aby kompleks Ripsa nie wybuchł pamięciowo
    if len(points) > max_points:
        indices = np.random.choice(len(points), max_points, replace=False)
        points = points[indices]
    elif len(points) == 0:
        # Zabezpieczenie przed pustymi obrazami
        points = np.array([[0,0], [1,1]]) 
        
    return points, img

def compute_betti_curve(diagram, thresholds):
    """Przekształca diagram persystencji w Krzywą Bettiego."""
    betti_curve = np.zeros(len(thresholds))
    if len(diagram) == 0:
        return betti_curve
        
    for birth, death in diagram:
        if np.isinf(death):
            death = thresholds[-1]
            
        # Zwiększamy licznik (numer Bettiego) dla wszystkich progów w przedziale [birth, death)
        alive_mask = (thresholds >= birth) & (thresholds < death)
        betti_curve[alive_mask] += 1
        
    return betti_curve

def extract_features():
    """Przechodzi przez dataset i ekstrahuje cechy TDA oraz cechy bazowe (Baseline)."""
    X_baseline = []
    X_tda = []
    y = []
    
    thresholds = np.linspace(0, MAX_FILTRATION, FILTRATION_STEPS)
    
    for label_idx, class_name in enumerate(CLASSES):
        class_path = os.path.join(DATA_DIR, class_name)
        if not os.path.exists(class_path):
            continue
            
        img_paths = glob(os.path.join(class_path, "*.*"))
        print(f"Przetwarzanie {class_name} ({len(img_paths)} obrazów)...")
        
        for img_path in img_paths:
            res = extract_point_cloud(img_path)
            if res is None:
                continue
            points, img_resized = res
            
            # --- 1. Cechy Bazowe (Baseline) ---
            # Spłaszczony i zredukowany obraz do porównania
            X_baseline.append(img_resized.flatten())
            y.append(label_idx)
            
            # --- 2. Cechy Topologiczne (TDA) ---
            # Tutaj pod spodem działa zoptymalizowany C++
            diagrams = ripser(points, maxdim=1)['dgms']
            
            H0_diagram = diagrams[0]
            H1_diagram = diagrams[1]
            
            betti_0 = compute_betti_curve(H0_diagram, thresholds)
            betti_1 = compute_betti_curve(H1_diagram, thresholds)
            
            # Łączymy krzywe Bettiego wymiaru 0 i 1 w jeden wektor cech
            tda_features = np.concatenate([betti_0, betti_1])
            X_tda.append(tda_features)

    return np.array(X_baseline), np.array(X_tda), np.array(y)

# --- WYKONANIE I PORÓWNANIE ---
print("Rozpoczynanie ekstrakcji cech...")
X_base, X_tda, y = extract_features()

if len(y) == 0:
    print("Nie znaleziono obrazów. Sprawdź ścieżki.")
else:
    # Wstępna redukcja wymiarowości dla Baseline (żeby RF miał równe szanse)
    pca = PCA(n_components=100)
    X_base_pca = pca.fit_transform(X_base)
    
    # Kombinacja cech: Baseline + TDA
    X_combined = np.hstack((X_base_pca, X_tda))
    
    # Podział na zbiory
    indices = np.arange(len(y))
    idx_train, idx_test = train_test_split(indices, test_size=0.25, random_state=42, stratify=y)
    
    # --- MODEL 1: Tylko Baseline ---
    rf_base = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_base.fit(X_base_pca[idx_train], y[idx_train])
    pred_base = rf_base.predict(X_base_pca[idx_test])
    acc_base = accuracy_score(y[idx_test], pred_base)
    
    # --- MODEL 2: Tylko TDA (Krzywe Bettiego) ---
    rf_tda = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_tda.fit(X_tda[idx_train], y[idx_train])
    pred_tda = rf_tda.predict(X_tda[idx_test])
    acc_tda = accuracy_score(y[idx_test], pred_tda)
    
    # --- MODEL 3: Baseline + TDA ---
    rf_combined = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_combined.fit(X_combined[idx_train], y[idx_train])
    pred_combined = rf_combined.predict(X_combined[idx_test])
    acc_combined = accuracy_score(y[idx_test], pred_combined)
    
    print("\n=== WYNIKI KLASYFIKACJI (Accuracy) ===")
    print(f"1. Baseline (Tylko cechy obrazu - PCA):  {acc_base:.4f}")
    print(f"2. Tylko TDA (Krzywe Bettiego H0 i H1):  {acc_tda:.4f}")
    print(f"3. Baseline + TDA (Złączone wektory):    {acc_combined:.4f}")

Rozpoczynanie ekstrakcji cech...
Przetwarzanie Amoeba (72 obrazów)...
Przetwarzanie Euglena (168 obrazów)...
Przetwarzanie Hydra (76 obrazów)...
Przetwarzanie Paramecium (152 obrazów)...
Przetwarzanie Rod_bacteria (85 obrazów)...
Przetwarzanie Spherical_bacteria (86 obrazów)...


libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50


Przetwarzanie Spiral_bacteria (75 obrazów)...
Przetwarzanie Yeast (75 obrazów)...

=== WYNIKI KLASYFIKACJI (Accuracy) ===
1. Baseline (Tylko cechy obrazu - PCA):  0.2677
2. Tylko TDA (Krzywe Bettiego H0 i H1):  0.2677
3. Baseline + TDA (Złączone wektory):    0.3687


In [6]:
import os
import cv2
import numpy as np
from glob import glob
import gudhi
from gudhi.representations import Landscape
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA

# --- KONFIGURACJA ---
DATA_DIR = "./Micro_Organism"  # Aktualny folder, w którym są subfoldery (Amoeba, Euglena, itd.)
CLASSES = [
    "Amoeba", "Euglena", "Hydra", "Paramecium", 
    "Rod_bacteria", "Spherical_bacteria", "Spiral_bacteria", "Yeast"
]
MAX_POINTS = 300  
MAX_FILTRATION = 20.0  

# Parametry dla Krajobrazów Persystentnych
NUM_LANDSCAPES = 5
RESOLUTION = 100

def extract_point_cloud(image_path, max_points=MAX_POINTS):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    
    img = cv2.resize(img, (64, 64))
    edges = cv2.Canny(img, threshold1=100, threshold2=200)
    
    y, x = np.where(edges > 0)
    points = np.column_stack((x, y))
    
    if len(points) > max_points:
        indices = np.random.choice(len(points), max_points, replace=False)
        points = points[indices]
    elif len(points) == 0:
        points = np.array([[0,0], [1,1]]) 
        
    return points, img

def extract_features():
    X_baseline = []
    X_tda = []
    y = []
    
    # Inicjalizacja wektoryzatora z Gudhi
    # Będzie on wyliczał 5 warstw krajobrazu, próbkowanych w 100 punktach
    landscape_transformer = Landscape(
        num_landscapes=NUM_LANDSCAPES, 
        resolution=RESOLUTION, 
        sample_range=[0, MAX_FILTRATION]
    )
    
    for label_idx, class_name in enumerate(CLASSES):
        class_path = os.path.join(DATA_DIR, class_name)
        if not os.path.exists(class_path):
            continue
            
        img_paths = glob(os.path.join(class_path, "*.*"))
        print(f"Przetwarzanie {class_name} ({len(img_paths)} obrazów)...")
        
        for img_path in img_paths:
            res = extract_point_cloud(img_path)
            if res is None:
                continue
            points, img_resized = res
            
            X_baseline.append(img_resized.flatten())
            y.append(label_idx)
            
            # --- TDA: Gudhi Rips Complex ---
            rips_complex = gudhi.RipsComplex(points=points, max_edge_length=MAX_FILTRATION)
            simplex_tree = rips_complex.create_simplex_tree(max_dimension=2)
            
            # Obliczenie homologii persystentnej (domyślnie szybki algorytm kohomologiczny)
            simplex_tree.compute_persistence()
            
            # Pobranie interwałów [birth, death] dla H0 i H1
            # Gudhi filtruje nieskończone interwały w module reprezentacji, 
            # ale dla pewności usuwamy je z H0 (komponenty spójne żyjące w nieskończoność)
            intervals_H0 = simplex_tree.persistence_intervals_in_dimension(0)
            intervals_H0 = intervals_H0[intervals_H0[:, 1] != np.inf] if len(intervals_H0) > 0 else np.empty((0, 2))
            
            intervals_H1 = simplex_tree.persistence_intervals_in_dimension(1)
            
            # Transformacja do postaci Krajobrazów
            # transformer oczekuje listy diagramów, więc przekazujemy je w nawiasach []
            land_H0 = landscape_transformer.fit_transform([intervals_H0])[0] if len(intervals_H0) > 0 else np.zeros(NUM_LANDSCAPES * RESOLUTION)
            land_H1 = landscape_transformer.fit_transform([intervals_H1])[0] if len(intervals_H1) > 0 else np.zeros(NUM_LANDSCAPES * RESOLUTION)
            
            # Łączymy wektory H0 i H1
            tda_features = np.concatenate([land_H0, land_H1])
            X_tda.append(tda_features)

    return np.array(X_baseline), np.array(X_tda), np.array(y)

# --- WYKONANIE I PORÓWNANIE ---
print("Rozpoczynanie ekstrakcji cech z wykorzystaniem Persistent Landscapes...")
X_base, X_tda, y = extract_features()

X_tda = np.nan_to_num(X_tda, nan=0.0, posinf=0.0, neginf=0.0)

if len(y) > 0:
    pca = PCA(n_components=100)
    X_base_pca = pca.fit_transform(X_base)
    
    X_combined = np.hstack((X_base_pca, X_tda))
    
    indices = np.arange(len(y))
    idx_train, idx_test = train_test_split(indices, test_size=0.25, random_state=42, stratify=y)
    
    rf_base = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_base.fit(X_base_pca[idx_train], y[idx_train])
    acc_base = accuracy_score(y[idx_test], rf_base.predict(X_base_pca[idx_test]))
    
    rf_tda = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_tda.fit(X_tda[idx_train], y[idx_train])
    acc_tda = accuracy_score(y[idx_test], rf_tda.predict(X_tda[idx_test]))
    
    rf_combined = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_combined.fit(X_combined[idx_train], y[idx_train])
    acc_combined = accuracy_score(y[idx_test], rf_combined.predict(X_combined[idx_test]))
    
    print("\n=== WYNIKI KLASYFIKACJI (Accuracy) ===")
    print(f"1. Baseline (Tylko cechy obrazu - PCA):  {acc_base:.4f}")
    print(f"2. Tylko TDA (Persistent Landscapes H0, H1):  {acc_tda:.4f}")
    print(f"3. Baseline + TDA (Złączone wektory):    {acc_combined:.4f}")

Rozpoczynanie ekstrakcji cech z wykorzystaniem Persistent Landscapes...
Przetwarzanie Amoeba (72 obrazów)...


/home/bamichal/miniforge3/envs/tda_stable/lib/python3.11/site-packages/gudhi/representations/vector_methods.py:271: RuntimeWarning: invalid value encountered in subtract
  heights[None, :] - np.abs(x_values[:, None] - midpoints[None, :]), 0


Przetwarzanie Euglena (168 obrazów)...
Przetwarzanie Hydra (76 obrazów)...
Przetwarzanie Paramecium (152 obrazów)...
Przetwarzanie Rod_bacteria (85 obrazów)...
Przetwarzanie Spherical_bacteria (86 obrazów)...


libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50


Przetwarzanie Spiral_bacteria (75 obrazów)...
Przetwarzanie Yeast (75 obrazów)...

=== WYNIKI KLASYFIKACJI (Accuracy) ===
1. Baseline (Tylko cechy obrazu - PCA):  0.2727
2. Tylko TDA (Persistent Landscapes H0, H1):  0.2727
3. Baseline + TDA (Złączone wektory):    0.3333
